# 3月7日17:50的回测与结果分析

## 回测设计  

这一次回测使用了全新的回测代码，对回测的逻辑进行了完全更新。  

这个版本的回测采用**随机种子机制**来控制环境初始化。每一轮测试都会使用不同的种子生成不同的游戏环境，确保每一次测试的游戏都是不同的，能更好的检测模型的泛化能力，测试结果也更具代表性。  

其他依旧沿用上一版本，使用**确定性策略**，每一步直接选择模型决策概率最高的动作执行，直接评估决策能力。  

在每一轮测试过程中，记录以下信息：  
- 总奖励（Reward）：整个 episode 的累计奖励  
- 步数（Steps）：从开始到结束所执行的动作数量  
- 成功标记（Success）：是否成功着陆  
- 坠毁标记（Crash）：是否发生坠毁  
- 超时标记（Timeout）：是否因为步数限制而结束  

并且生成可视化统计图。  

### 游戏分数机制补充  

对于目标游戏，通常官方过关分数为200分，优秀分数在250-300分，300-320分为顶尖分数，极限分数一般在320分以上  

## 训练概要  

本次回测的模型采用Actor–Critic架构  

神经网络结构如下：  

state_dim -> 128 -> ReLU -> 128 -> 256 -> RELU -> 256 -> 128 -> ReLU  

-> action_dim（决策层）  
-> 1(价值评估层)  

训练代码超参数如下：

In [ ]:
# 超参数
LR = 3e-4            # 学习率
gamma = 0.99         # 折扣因子
eps_clip = 0.2       # PPO裁剪系数
value_coef = 0.5     # critic损失权重
entropy_coef = 0.01  # 熵奖励权重(鼓励探索)
updates = 1000       # PPO 更新次数
# PPO稳定参数
gae_lambda = 0.95            # GAE lambda 参数（取值范围 [0, 1]，控制 bias-variance 权衡）
rollout_steps = 2048         # 每次更新前收集的步数
ppo_update_epochs = 10       # 同一批数据重复训练次数
mini_batch_size = 256        # mini-batch 大小
max_grad_norm = 0.5          # 梯度裁剪

### 训练结果  

以下是训练结果可视化：  

![cat](lib/yh2/befor/reward_curve.png)  

![cat](lib/yh2/befor/loss_curve.png)  

![cat](lib/yh2/befor/entropy_curve.png)

### 训练结果分析  

#### 回报分析  

如图所示，训练中Reward回报从-300左右逐步上升到250+并且稳定在200-260之间，可以认为模型已经学会了着陆策略，并且策略总体上是收敛的。  

但是，回报曲线存在较大的方差，图中蓝色曲线存在大量-400到-600的极端值，说明策略可能并不稳定  

移动平均（橙色曲线）从-200稳定提升到250左右，说明平均决策都是正确的  

#### PPO loss分析  

从 PPO 的 Loss 曲线来看，Actor Loss（蓝线）始终接近 0，整体策略变化较为平稳，这符合 PPO 在训练的常见表现  

与之相比，Critic Loss（橙线）明显偏高，且在整个训练过程中存在大量尖峰，很多时候落在 100~200 区间，甚至多次出现 200+、400+、900 左右的异常值。这说明价值网络对回报的拟合误差始终较大，训练过程中存在显著波动，价值层学习稳定性较差  

Total Loss（绿线）整体走势与 Critic Loss 高度一致，表明总损失主要由价值误差主导。虽然从整体趋势看，Loss 有一定下降，说明模型并非完全没有学习能力，但频繁出现的大幅波动说明这种学习过程质量不高，稳定性较差  

综合来看，当前模型表面上w在持续训练，但其价值估计部分始终存在较强的不确定性。这是一个值得关注的问题  

#### Entropy分析  

随机熵从1.4左右下降到0.5-0.7，说明策略已经从高度随机转变为有一定确定性，但是依然没有完全收敛，依然有明显的随机性。这也解释了为什么回报曲线中有大量偶发的极端值  

#### 综合说明  

根据训练结果，可以分析出模型目前存在几个关键的问题  

1. 价值层学习不稳定：Critic Loss严重偏高，学习非常不稳定  
2. 决策随机熵偏高：策略目前性较强，导致部分情况产生极低的回报  
3. 回报方差很大：在部分状态时策略没有完成学习  


## 回测结果分析  

![cat](lib/yh2/befor/evaluation_summary.png)  

### 策略成功但是不稳定  

回测结果标明，大量reward落在200以上，但存在部分落在0左右，还有少数极端负值，综合成功率不到50%。这个现象说明模型策略已经学会了如何着陆，但是没有学会如何保证稳定着陆，即鲁棒性极差。  

### 对两种失败模式的分析  

#### Crash失败  

即坠毁，其特征为step在250到400之间，reward在-200到-400之间。通常是飞船已经接近地面，甚至触地，但是姿态或者速度控制失败。  

这表明模型在某些状态下没有完成学习，有极端失败存在。  

#### TimeOut失败  

即超时，其特征为step超过1000，reward在0到100之间。通常是飞船长时间悬停不降落，说明策略在部分状态下更倾向于保守控制以避免坠毁。  

这个现象说明模型在部分状态下学会了避免惩罚而不是完成任务。  

### Reward分布分析  

回报分布标明，模型在某些状态下有了高置信策略，但是在一些其他状态几乎没有学习到任何策略。这意味着模型对部分状态学习不足。  

## 综合说明  

训练结果与回测结果现象分析如下  

训练现象：  
1. 价值层loss非常大  
2. 随机熵较高  
3. 回报方差较大  

回测现象：  
1. 着陆阶段不稳定  
2. 策略部分状态坠毁  
3. 策略部分状态悬停  

二者反映的问题高度一致  

由此可以简单推断策略结构如下：

状态空间 S  
│  
├── 大部分区域：学会着陆  
│  
├── 一些区域：策略随机  
│  
└── 一些区域：选择悬停  

因此当前模型已经形成可行策略，但策略鲁棒性不足。

## 对模型的优化  

基于刚刚的问题分析，其实模型已经具备了学习这个任务的能力，所以现在的问题是训练稳定性的问题。因此这次优化不会聚焦于模型结构和具体实现。  

#### 分离actor/critic学习率  

给critic更小的学习率，平衡其影响  

#### 将critic的MSE更换为Huber Loss  

MSE（均方误差）算法对极端值较为敏感，若batch中混入极端值，容易造成critic loss爆炸。因此更换为Huber Loss，一种结合了MSE和MAE（绝对误差）的损失函数，降低极端值对训练的影响  

#### 【退回】增加对returns的标准化  

【此修改导致尺度信号不匹配】  

原有代码已经对advantages进行了标准化，但没有对returns标准化。标准化后将减小critic拟合压力，缓解大Loss  

#### 【退回】增大mini_batch_size  

【大batch降低模型精细学习能力，末端控制不足】

batch增大会让梯度估计更稳，影响未知，姑且试一试  

#### 降低ppo_update_epochs  

避免同一批rollout被反复学习过多次，造成过拟合  

#### 增加actor/critic分支复杂性  

让actor和critic分支能分别学到更多各自的内容，避免共享层过多导致噪声影响  

#### 加入 entropy 衰减  

让随机熵权重随着训练进行衰减，具体效果为前期鼓励探索，后期动作更加确定  

## 优化后结果回测  

优化产生了显著作用，并且在回测中得以体现。  

测试局数          : 30  
平均奖励          : 281.49  
奖励标准差        : 18.04  
最高奖励          : 313.08  
最低奖励          : 246.02  
平均步数          : 213.13  
成功次数          : 30  
坠毁次数          : 0  
超时次数          : 0  
成功率            : 100.00%  
坠毁率            : 0.00%  
超时率            : 0.00%  

![cat](lib/yh2/after/reward_curve.png)  

![cat](lib/yh2/after/loss_curve.png)  

![cat](lib/yh2/after/entropy_curve.png)

![cat](lib/yh2/after/evaluation_summary.png)